# RoSE routing pilot — one 4B model over the 150-question sample

Runs `chartqapro/routing/pilot_runner.py` on Kaggle (T4, 4-bit NF4): the repo's
real RoSE pipeline with a 4B VLM, over the committed seed-499 pilot sample.
**One model per session** — run this notebook once with `MODEL_KEY = "gemma3-4b"`
and once with `MODEL_KEY = "qwen3.5-4b"`.

Before the Gemma session:
1. Accept the license at https://huggingface.co/google/gemma-3-4b-it (gated model).
2. Add your HF token as a Kaggle secret named `HF_TOKEN`
   (Add-ons → Secrets) and enable it for this notebook.
3. Settings: Accelerator **GPU T4 x2** (or P100), Internet **ON**.

Workflow: run all cells with `DRY_RUN = True` (≈10 min sanity check), then set
`DRY_RUN = False` and re-run **only the last code cell** — the checkpoint keeps
the dry-run answers, so nothing is wasted.


In [ ]:
# ── config ───────────────────────────────────────────────────────
MODEL_KEY = "gemma3-4b"       # "gemma3-4b" | "qwen3.5-4b"  — one per session
DRY_RUN   = True              # True = first 5 sampled questions only
QUANT     = "nf4"             # the pilot spec; "fp4" only to mirror the old 7B sweep

REPO_URL  = "https://github.com/ktahsinr/RoSE.git"
REPO_REF  = "feature/model-routing"

# To resume a killed session: upload the previous session's
# pilot_<model>_checkpoint.json as a Kaggle dataset and point at it here.
RESUME_FROM = None            # e.g. "/kaggle/input/rose-pilot-ckpt/pilot_gemma3-4b_checkpoint.json"

from pathlib import Path
import sys
try:
    sys.stdout.reconfigure(line_buffering=True)   # live logs in Kaggle's pane
except (AttributeError, ValueError):
    pass

ON_KAGGLE   = Path("/kaggle").exists()
WORK        = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
SCRATCH     = Path("/kaggle/temp")    if ON_KAGGLE else WORK / "scratch"
CODE_DIR    = SCRATCH / "RoSE"
IMAGES_DIR  = SCRATCH / "images"
RESULTS_DIR = WORK / "results" / "pilot"          # published with the notebook
DATASET_JSON = SCRATCH / "chartqapro_factoid_pilot.json"
OUT_FILE     = RESULTS_DIR / f"pilot_{MODEL_KEY}.json"

for d in (SCRATCH, IMAGES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"model={MODEL_KEY}  dry_run={DRY_RUN}  quant={QUANT}")


In [ ]:
# ── code + pinned libraries ──────────────────────────────────────
import subprocess
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                    REPO_URL, str(CODE_DIR)], check=True)
else:
    # a kernel restart keeps /kaggle/temp — refresh so fixes pushed to the
    # branch mid-session are picked up instead of running stale code
    subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1",
                    "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "reset", "--hard",
                    "FETCH_HEAD"], check=True)
print("repo @", subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(CODE_DIR / "requirements-pinned.txt")], check=True)
# NOTE: transformers==4.51.3 (the sweep pin) supports Gemma 3. If
# Qwen/Qwen3.5-4B fails to load with an unknown-architecture error, install a
# newer transformers here for that session — the run's *_meta.json records the
# exact version either way, so the report stays honest.

sys.path.insert(0, str(CODE_DIR / "chartqapro"))
import transformers
print("transformers:", transformers.__version__)


In [ ]:
# ── HuggingFace auth + gated-access check ───────────────────────
# Gemma 3 is GATED: this cell must end with "access to google/gemma-3-4b-it
# verified" or the model download will 401. If it raises, fix the named step
# and re-run just this cell.
from routing.config import MODELS
HF_ID = MODELS[MODEL_KEY]["hf_id"]

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"could not read Kaggle secret HF_TOKEN: {e}")

if token:
    from huggingface_hub import login
    login(token=token)
    print("HF login ok")

from huggingface_hub import model_info
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError
try:
    info = model_info(HF_ID)
    print(f"access to {HF_ID} verified (gated={getattr(info, 'gated', False)})")
except GatedRepoError:
    raise SystemExit(
        f"\nNO ACCESS to the gated model {HF_ID}.\n"
        + ("→ Your token works but the LICENSE is not accepted (or still pending):\n"
           f"  visit https://huggingface.co/{HF_ID} logged in as the token's account,\n"
           "  accept the license, then re-run this cell.\n" if token else
           "→ No HF token reached this session:\n"
           "  1. hf.co → Settings → Access Tokens → create a READ token\n"
           "  2. Kaggle → Add-ons → Secrets → add it with label HF_TOKEN\n"
           "  3. TICK THE CHECKBOX attaching the secret to THIS notebook\n"
           "  then re-run this cell.\n"))
except RepositoryNotFoundError:
    raise SystemExit(f"{HF_ID} not found — with a gated repo this can also mean "
                     "the token has no access; check token + license.")


In [ ]:
# ── build the pilot dataset ──────────────────────────────────────
# Ids are assigned by FACTOID ORDINAL in split order (factoid-00000 …), the
# convention of the cached 7B results the sample was drawn from. Only the 150
# sampled questions get their images decoded and written.
import io, json, ast
from collections import Counter
from PIL import Image
from datasets import load_dataset

sample = json.loads((CODE_DIR / "chartqapro/results/pilot/sample_ids.json").read_text())
sample_ids = set(sample["ids"])
print(f"sample: n={sample['n']} seed={sample['seed']} source={sample['source']}")

def _gold(v):
    if isinstance(v, (list, tuple)):
        return [str(x) for x in v]
    s = str(v).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            if isinstance(p, (list, tuple)):
                return [str(x) for x in p]
        except Exception:
            pass
    return s

def _save_image(field, path):
    if isinstance(field, dict) and field.get("bytes"):
        img = Image.open(io.BytesIO(field["bytes"]))
    elif isinstance(field, (bytes, bytearray)):
        img = Image.open(io.BytesIO(field))
    elif isinstance(field, Image.Image):
        img = field
    else:
        return str(field)
    img.convert("RGB").save(path, format="PNG")
    return str(path)

ds = load_dataset("ahmed-masry/ChartQAPro", split="test")
qtypes = [str(t).strip().lower() for t in ds["Question Type"]]
fact_idx = [i for i, t in enumerate(qtypes) if t == "factoid"]
print(f"factoid questions in split: {len(fact_idx)}")

records = []
for idx, i in enumerate(fact_idx):           # idx = factoid ordinal = id
    qid = f"factoid-{idx:05d}"
    if qid not in sample_ids:
        continue
    row = ds[i]
    records.append({
        "id":            qid,
        "question":      str(row.get("Question", "")).strip(),
        "choices":       None,
        "answer":        _gold(row.get("Answer")),
        "question_type": "factoid",
        "image":         _save_image(row.get("image"), IMAGES_DIR / f"{idx:05d}.png"),
        "orig_type":     str(row.get("Question Type", "")),
        "year":          row.get("Year"),
    })

assert len(records) == sample["n"], f"built {len(records)} rows, expected {sample['n']}"
DATASET_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2))
print(f"{len(records)} sampled questions + images → {DATASET_JSON}")


In [ ]:
# ── sanity: the pilot must run the final-week RoSE config ────────
import rose_chartqapro as rose
assert (rose.M_PATHS, rose.K_DEMONSTRATIONS, rose.LAMBDA, rose.TEMPERATURE) \
       == (5, 3, 1.2, 0.8), "rose_chartqapro defaults drifted from the sweep config"
assert not rose.PAPER_FAITHFUL
rose.IMAGES_DIR = str(IMAGES_DIR)
print(f"m={rose.M_PATHS}  k={rose.K_DEMONSTRATIONS}  λ={rose.LAMBDA}  T={rose.TEMPERATURE}")
for name, on in rose.active_extensions().items():
    print(f"   {'✓' if on else '·'} {name}")

if RESUME_FROM:
    import shutil
    ckpt = OUT_FILE.with_name(OUT_FILE.stem + "_checkpoint.json")
    shutil.copy(RESUME_FROM, ckpt)
    print(f"resume checkpoint staged → {ckpt}")


In [ ]:
# ── load the model once (reused by dry run AND real run) ─────────
from routing.pilot_runner import load_vlm, run_pilot
from sentence_transformers import SentenceTransformer

model, processor, revision = load_vlm(MODEL_KEY, quant=QUANT)
embedder = SentenceTransformer(rose.EMBED_MODEL)
print(f"loaded {MODEL_KEY} @ revision {revision}")


In [ ]:
# ── run ──────────────────────────────────────────────────────────
# DRY_RUN answers the first 5 sampled questions (~10 min). Flip DRY_RUN to
# False in the config cell and re-run JUST THIS CELL for the full 150 — the
# checkpoint keeps the 5 dry-run answers, so they are not redone.
results = run_pilot(
    MODEL_KEY, DATASET_JSON, CODE_DIR / "chartqapro/results/pilot/sample_ids.json",
    OUT_FILE, quant=QUANT,
    model=model, processor=processor, embedder=embedder,
    limit=5 if DRY_RUN else None,
)

ok = [r for r in results if r.get("method") != "error"]
lat = [r["latency_s"] for r in ok if r.get("latency_s")]
print(f"\nanswered: {len(ok)}  errors: {len(results) - len(ok)}")
if lat:
    print(f"accuracy: {sum(r['is_correct'] for r in ok) / len(ok) * 100:.1f}%"
          f"   mean latency: {sum(lat) / len(lat):.1f}s"
          f"   projected 150-question time: {sum(lat) / len(lat) * 150 / 3600:.1f}h")
print(f"\ndownload from the Output tab: {OUT_FILE.name}, "
      f"{OUT_FILE.stem}_meta.json, {OUT_FILE.stem}_checkpoint.json")


## After the session

1. **Save Version → Save & Run All** is not required — interactive runs keep
   `/kaggle/working`; download the three files from the Output pane
   (`pilot_<model>.json`, `_meta.json`, `_checkpoint.json`).
2. Drop `pilot_<model>.json` and `pilot_<model>_meta.json` into
   `chartqapro/results/pilot/` in the repo and commit (the folder is
   git-tracked). The checkpoint only matters if the run died early.
3. Back-fill the pins: copy the exact versions from `pilot_<model>_meta.json`
   into `requirements-pinned.txt`, and the resolved `revision` into
   `chartqapro/routing/config.py`.
4. Once **both** 4B models are done, everything else is CPU, on your laptop:

```bash
cd chartqapro
python -m routing.pool \
    --results gemma3-4b=results/pilot/pilot_gemma3-4b.json \
              qwen3.5-4b=results/pilot/pilot_qwen3.5-4b.json \
              qwen2.5-vl-7b=results/rose_factoid.json \
    --sample results/pilot/sample_ids.json \
    --out results/pilot/routing_pool.json
python -m routing.evaluate_pilot results/pilot/routing_pool.json \
    --out results/pilot/pilot_report.json
```
